In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
data_path = Path("../data/heart.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path.resolve()}")

df = pd.read_csv(data_path).dropna().copy()

FEATURE_COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal"
]
CATEGORICAL_COLS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
NUMERICAL_COLS = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_COLS]

X_raw = df[FEATURE_COLUMNS]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("scaler", StandardScaler())]), NUMERICAL_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
    ]
)

X_processed = preprocessor.fit_transform(X_raw)
if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

print("Processed shape:", X_processed.shape)

Processed shape: (1025, 30)


In [3]:
sorted_idx = np.argsort(df["age"].values)
X_seq = X_processed[sorted_idx]
df_seq = df.iloc[sorted_idx].reset_index(drop=True)

seq_len = 12
lengths = []
start = 0
while start < len(X_seq):
    chunk = min(seq_len, len(X_seq) - start)
    lengths.append(chunk)
    start += chunk

hmm_model = GaussianHMM(
    n_components=3,
    covariance_type="diag",
    n_iter=400,
    random_state=42,
)
hmm_model.fit(X_seq, lengths)
hidden_states = hmm_model.predict(X_seq, lengths)

risk_df = df_seq.copy()
risk_df["hidden"] = hidden_states
risk_df["risk_score"] = (
    0.20 * (risk_df["age"] / risk_df["age"].max())
    + 0.20 * (risk_df["chol"] / risk_df["chol"].max())
    + 0.20 * (risk_df["trestbps"] / risk_df["trestbps"].max())
    + 0.25 * (risk_df["oldpeak"] / (risk_df["oldpeak"].max() + 1e-8))
    + 0.15 * (1 - (risk_df["thalach"] / risk_df["thalach"].max()))
)

ordered = risk_df.groupby("hidden")["risk_score"].mean().sort_values().index.tolist()
labels = ["Healthy", "At Risk", "Diseased"]
state_name_map = {int(st): labels[min(i, 2)] for i, st in enumerate(ordered)}

print("State mapping:", state_name_map)

State mapping: {0: 'Healthy', 2: 'At Risk', 1: 'Diseased'}


In [4]:
artifact = {
    "model": hmm_model,
    "preprocessor": preprocessor,
    "feature_columns": FEATURE_COLUMNS,
    "state_name_map": state_name_map,
}

model_path = Path("model.pkl")
joblib.dump(artifact, model_path)
print("Saved:", model_path.resolve())

Saved: C:\CODING\PYTHON\ML\MarkovModel\model.pkl
